# PARTIE 2 -- Optimisation et bilan d'architecture (après-midi)

## 2.1 Anatomie d'un plan d'exécution Spark

Quand vous soumettez une requête DataFrame ou SQL, Spark ne l'exécute pas
immédiatement. Il la fait d'abord passer par quatre phases de planification.

```
Code Python / SQL
       │
       ▼
  Plan non résolu (Unresolved Logical Plan)
  "Résolution" : vérification des noms de colonnes et des types
       │
       ▼
  Plan logique résolu (Resolved Logical Plan)
  "Analyse" : règles d'optimisation algébrique (push-down, élimination)
       │
       ▼
  Plan logique optimisé (Optimized Logical Plan)
  Catalyst applique ~70 règles de réécriture
       │
       ▼
  Plan(s) physique(s) (Physical Plans)
  Plusieurs stratégies sont énumérées (SortMerge, Broadcast, Hash...)
       │
       ▼
  Plan physique sélectionné (Selected Physical Plan)
  Tungsten génère le bytecode JVM optimisé (whole-stage codegen)
       │
       ▼
  Exécution distribuée
```

`explain(mode="formatted")` vous montre ces plans à la demande.


In [28]:
# Cette partie continue le carnet précédent (même noyau) : df, FEATURES, df_clusters, le modèle
# sauvegardé, etc. sont déjà en mémoire. On prépare seulement le recueil des mesures.
import contextlib, io
import requests

MESURES = {}      # rempli au fil des cellules, puis enregistré dans mesures.json (bilan final)

def capturer_plan(donnees) -> str:
    """Renvoie le plan physique d'un DataFrame sous forme de texte.

    Args:
        donnees: DataFrame Spark (le plan n'est pas exécuté, seulement calculé).

    Returns:
        Le texte affiché par ``explain(mode="simple")``.
    """
    tampon = io.StringIO()
    with contextlib.redirect_stdout(tampon):
        donnees.explain(mode="simple")
    return tampon.getvalue()

In [29]:
# Illustration : requête simple -- que fait Catalyst ?
df_exemple = (
    df
    .filter(col("annee") == 2021)
    .filter(col("heure").between(7, 9))
    .groupBy("station_id")
    .agg(spark_avg("taux_occupation").alias("taux_pointe"))
    .filter(col("taux_pointe") < 0.2)   # filtre APRES agrégation
)

print("=== Plan logique (ce que vous avez écrit) ===")
df_exemple.explain(mode="simple")

=== Plan logique (ce que vous avez écrit) ===


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter (isnotnull(taux_pointe#264411) AND (taux_pointe#264411 < 0.2))
   +- HashAggregate(keys=[station_id#48], functions=[avg(taux_occupation#56)])
      +- Exchange hashpartitioning(station_id#48, 8), ENSURE_REQUIREMENTS, [plan_id=19546]
         +- HashAggregate(keys=[station_id#48], functions=[partial_avg(taux_occupation#56)])
            +- Project [station_id#48, taux_occupation#56]
               +- Filter ((isnotnull(heure#59) AND (heure#59 >= 7)) AND (heure#59 <= 9))
                  +- FileScan parquet [station_id#48,taux_occupation#56,heure#59,annee#66,mois#67] Batched: true, DataFilters: [isnotnull(heure#59), (heure#59 >= 7), (heure#59 <= 9)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/home/jovyan/data/output/delta/disponibilite], PartitionFilters: [isnotnull(annee#66), (annee#66 <= 2021), (annee#66 = 2021)], PushedFilters: [IsNotNull(heure), GreaterThanOrEqual(heure,7), LessThanOrEqual(heure,

In [30]:
print("=== Plan physique optimisé (ce que Spark exécutera réellement) ===")
df_exemple.explain(mode="formatted")
# Observez :
# 1. Spark pousse le filtre "annee = 2021" AVANT la lecture (PartitionFilter)
# 2. Il applique le filtre "heure BETWEEN 7 AND 9" pendant le scan (DataFilter)
# 3. L'agrégation utilise HashAggregate (plus rapide que SortAggregate)
# 4. Il applique le filtre final "taux_pointe < 0.2" APRÈS l'agrégation

=== Plan physique optimisé (ce que Spark exécutera réellement) ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Filter (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [station_id#48, taux_occupation#56, heure#59, annee#66, mois#67]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/jovyan/data/output/delta/disponibilite]
PartitionFilters: [isnotnull(annee#66), (annee#66 <= 2021), (annee#66 = 2021)]
PushedFilters: [IsNotNull(heure), GreaterThanOrEqual(heure,7), LessThanOrEqual(heure,9)]
ReadSchema: struct<station_id:int,taux_occupation:double,heure:int>

(2) Filter
Input [5]: [station_id#48, taux_occupation#56, heure#59, annee#66, mois#67]
Condition : ((isnotnull(heure#59) AND (heure#59 >= 7)) AND (heure#59 <= 9))

(3) Project
Output [2]: [station_id#48, taux_occupation#56]
Input [5]: [station_id#48, taux_occupation#56, he

---
## 2.2 Identifier et corriger un data skew

Le **data skew** (déséquilibre de données) est l'une des causes les plus
fréquentes de lenteur dans Spark. Il se produit quand les données ne sont
pas distribuées équitablement entre les partitions après un shuffle.

**Symptôme dans le Spark UI** : un stage dont la durée totale est dominée
par 1 ou 2 tâches alors que les autres finissent en quelques secondes.

### Simulation d'un skew


In [31]:
# Simulation d'un DataFrame avec skew artificiel
# Une station (la station 0) représente 90% des données
import random
random.seed(SEED)

n_normal = 100_000
n_skewed  = 900_000

df_normal  = spark.range(n_normal).withColumn(
    "station_id", (F.rand(seed=SEED) * 100).cast("int") + 1
)
df_dominant = spark.range(n_skewed).withColumn(
    "station_id", F.lit(0)
)
df_skewed = df_normal.union(df_dominant).withColumn(
    "valeur", F.rand(seed=SEED)
)

print(f"DataFrame skewé : {df_skewed.count():,} lignes")
print("Distribution des 5 stations les plus fréquentes :")
df_skewed.groupBy("station_id").count().orderBy(F.desc("count")).show(5)


DataFrame skewé : 1,000,000 lignes
Distribution des 5 stations les plus fréquentes :


+----------+------+
|station_id| count|
+----------+------+
|         0|900000|
|         7|  1054|
|        57|  1052|
|        97|  1051|
|        81|  1051|
+----------+------+
only showing top 5 rows



In [32]:
# ── Approche naïve : groupBy direct ──────────────────────────────────────────
t0 = time.perf_counter()
df_skewed.groupBy("station_id").agg(spark_avg("valeur")).count()
t_naif = time.perf_counter() - t0
print(f"GroupBy naïf            : {t_naif:.2f} s")
MESURES["skew_groupby_naif_s"] = t_naif
print("  -> Observez dans le Spark UI : une tâche prend beaucoup plus longtemps")
print("     que les autres dans le stage du shuffle.")

GroupBy naïf            : 0.19 s
  -> Observez dans le Spark UI : une tâche prend beaucoup plus longtemps
     que les autres dans le stage du shuffle.


In [33]:
# ── Technique 1 : salting (ajout d'un sel aléatoire) ─────────────────────────
# On divise la clé dominante en N sous-groupes, on agrège partiellement,
# puis on supprime le sel et on agrège globalement.
N_SEL = 10

df_sale = df_skewed.withColumn(
    "cle_salee",
    F.concat(
        col("station_id").cast("string"),
        F.lit("_"),
        (F.rand(seed=SEED) * N_SEL).cast("int").cast("string")
    )
)

t0 = time.perf_counter()
# Agrégation partielle sur la clé salée
df_partiel = (
    df_sale
    .groupBy("cle_salee", "station_id")
    .agg(
        spark_avg("valeur").alias("moy_partielle"),
        F.count("*").alias("n_partielle")
    )
)
# Agrégation finale sur la clé originale (moyenne pondérée)
df_final_sale = (
    df_partiel
    .groupBy("station_id")
    .agg(
        (F.sum(col("moy_partielle") * col("n_partielle")) / F.sum("n_partielle"))
        .alias("moy_finale")
    )
)
df_final_sale.count()
t_sale = time.perf_counter() - t0

print(f"GroupBy avec salting     : {t_sale:.2f} s")
print(f"Gain                     : x{t_naif / t_sale:.1f}")
MESURES["skew_groupby_sale_s"] = t_sale
# Lecture : ici l'écart (~0,1 s) est dans le bruit de mesure : en mode local le salting apporte peu (il ajoute
# une seconde agrégation). Il prend son sens sur un cluster : là, la tâche qui traite la clé
# dominante devient le goulot, et le salage la répartit sur plusieurs exécuteurs. Le bon
# réflexe est de DIAGNOSTIQUER (Spark UI, onglet Stages : écart médiane / maximum des durées
# de tâches) avant d'appliquer une technique.

GroupBy avec salting     : 0.07 s
Gain                     : x2.8


In [34]:
# ── Technique 2 : broadcast de la petite table ────────────────────────────────
# Si l'une des tables d'une jointure est petite, on la broadcast
# pour éviter entièrement le shuffle de la grande table.

import time
df_grande = df.filter(col("annee") == 2021)
# Petite table : une catégorie par station, construite localement depuis le référentiel
# (ADAPTATION : le modèle numérotait range(1500) alors que nos identifiants vont de 1001 à 2402 ;
# la jointure aurait perdu la moitié des stations).
stations_pd = pd.read_csv(STATIONS_CSV, sep=";")
df_petite = spark.createDataFrame(
    stations_pd.assign(categorie=lambda d: "Catégorie " + (d["station_id"] % 4).astype(str))
    [["station_id", "categorie"]]
)

print(f"Grande table : {df_grande.count():,} lignes")
print(f"Petite table : {df_petite.count()} lignes")

# Sans broadcast : SortMergeJoin (shuffle des deux tables)
# Par défaut, Spark diffuse AUTOMATIQUEMENT toute table plus petite que
# spark.sql.autoBroadcastJoinThreshold (ici 50 Mo) : notre petite table serait donc déjà
# broadcastée sans rien demander, et la comparaison n'aurait aucun sens. On désactive
# temporairement ce seuil (-1) pour forcer le SortMergeJoin, puis on le restaure.
seuil_initial = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
jointure_smj = df_grande.join(df_petite, on="station_id", how="inner")
plan_smj = capturer_plan(jointure_smj)          # calculé tant que le seuil est désactivé
t0 = time.perf_counter()
n1 = jointure_smj.count()
t_smj = time.perf_counter() - t0
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", seuil_initial)

# Avec broadcast : BroadcastHashJoin (pas de shuffle de la grande table)
jointure_bcast = df_grande.join(F.broadcast(df_petite), on="station_id", how="inner")
plan_bcast = capturer_plan(jointure_bcast)
t0 = time.perf_counter()
n2 = jointure_bcast.count()
t_bcast = time.perf_counter() - t0

# Nombre de shuffles (« Exchange hashpartitioning ») lus dans les plans physiques
shuffles_smj   = plan_smj.count("Exchange hashpartitioning")
shuffles_bcast = plan_bcast.count("Exchange hashpartitioning")

print(f"\nSortMergeJoin    : {t_smj:.2f} s  ({n1:,} lignes)  -- {shuffles_smj} shuffle(s) : "
      f"{'SortMergeJoin' in plan_smj}")
print(f"BroadcastHashJoin: {t_bcast:.2f} s  ({n2:,} lignes)  -- {shuffles_bcast} shuffle(s) : "
      f"{'BroadcastHashJoin' in plan_bcast}")
print(f"Gain             : x{t_smj / t_bcast:.1f}")
assert n1 == n2, "Les deux jointures devraient donner le même nombre de lignes"
print()
print("Seuil automatique de broadcast (configurable) :")
print(f"  spark.sql.autoBroadcastJoinThreshold = "
      f"{spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes")

MESURES["broadcast"] = {"smj_s": t_smj, "broadcast_s": t_bcast, "gain": t_smj / t_bcast,
                         "shuffles_smj": shuffles_smj, "shuffles_broadcast": shuffles_bcast}

Grande table : 7,470,645 lignes


Petite table : 1402 lignes



SortMergeJoin    : 1.09 s  (7,470,645 lignes)  -- 2 shuffle(s) : True
BroadcastHashJoin: 0.37 s  (7,470,645 lignes)  -- 0 shuffle(s) : True
Gain             : x3.0

Seuil automatique de broadcast (configurable) :
  spark.sql.autoBroadcastJoinThreshold = 52428800 bytes


---
## 2.3 Partitionnement optimal

Le nombre de partitions a un impact direct sur les performances.
Trop peu : les tâches sont longues, les coeurs restent inactifs.
Trop de partitions : l'overhead de scheduling dépasse le gain du parallélisme.

**Règle empirique** : viser des partitions de 100 à 300 MB après décompression.
En mode local, aligner sur le nombre de coeurs disponibles.


In [35]:
import os

n_coeurs = os.cpu_count()
print(f"Coeurs disponibles : {n_coeurs}")
print(f"spark.sql.shuffle.partitions (actuel) : "
      f"{spark.conf.get('spark.sql.shuffle.partitions')}")

# ── Impact du nombre de partitions sur un calcul itératif ──────────────────
df_test_part = df.filter(col("annee") == 2021)
n_lignes     = df_test_part.count()
print(f"\nDataFrame de test : {n_lignes:,} lignes")

# Spark 3 active par défaut l'exécution adaptative (AQE), qui FUSIONNE automatiquement les petites
# partitions après un shuffle : le réglage de spark.sql.shuffle.partitions serait alors masqué et
# tous les essais donneraient le même résultat. On la désactive le temps de l'expérience.
aqe_initial = spark.conf.get("spark.sql.adaptive.enabled")
spark.conf.set("spark.sql.adaptive.enabled", "false")

temps_par_partitions = {}
for n_parts in [2, 4, 8, 16, 32]:
    spark.conf.set("spark.sql.shuffle.partitions", n_parts)
    t0 = time.perf_counter()
    (
        df_test_part
        .groupBy("station_id", "heure")
        .agg(spark_avg("taux_occupation"))
        .count()
    )
    t = time.perf_counter() - t0
    temps_par_partitions[n_parts] = round(t, 2)
    taille_part = n_lignes / n_parts
    print(f"  {n_parts:>3} partitions  ({taille_part:>8,.0f} lignes/partition) : {t:.2f} s")

# Restauration
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
spark.conf.set("spark.sql.adaptive.enabled", aqe_initial)
MESURES["partitions_shuffle_s"] = temps_par_partitions
# Dans le Spark UI (onglet Stages), le stage « après shuffle » de chaque essai compte exactement
# n_parts tâches : trop peu = tâches longues et coeurs inoccupés ; trop = surcoût d'ordonnancement.

Coeurs disponibles : 24
spark.sql.shuffle.partitions (actuel) : 8

DataFrame de test : 7,470,645 lignes


    2 partitions  (3,735,322 lignes/partition) : 0.45 s


    4 partitions  (1,867,661 lignes/partition) : 0.20 s
    8 partitions  ( 933,831 lignes/partition) : 0.13 s


   16 partitions  ( 466,915 lignes/partition) : 0.17 s
   32 partitions  ( 233,458 lignes/partition) : 0.15 s


In [36]:
# ── Repartitionnement explicite vs coalesce ───────────────────────────────────
# repartition(n) : shuffle complet, redistribution équilibrée
# coalesce(n)    : fusion de partitions adjacentes, sans shuffle
#                  (uniquement pour RÉDUIRE le nombre de partitions)

print("Partitions actuelles du DataFrame :", df.rdd.getNumPartitions())

t0 = time.perf_counter()
df_reparti = df.repartition(n_coeurs)
df_reparti.count()
t_reparti = time.perf_counter() - t0

t0 = time.perf_counter()
df_coalesce = df.coalesce(n_coeurs)
df_coalesce.count()
t_coalesce = time.perf_counter() - t0

print(f"repartition({n_coeurs}) : {t_reparti:.2f} s -- {df_reparti.rdd.getNumPartitions()} partitions")
print(f"coalesce({n_coeurs})    : {t_coalesce:.2f} s -- {df_coalesce.rdd.getNumPartitions()} partitions")
print()
MESURES["repartition_vs_coalesce_s"] = {"repartition": t_reparti, "coalesce": t_coalesce}
print("Règle :")
print("  Augmenter ou équilibrer les partitions -> repartition() (shuffle)")
print("  Réduire les partitions avant une écriture -> coalesce() (pas de shuffle)")

Partitions actuelles du DataFrame : 21


repartition(24) : 0.47 s -- 24 partitions


coalesce(24)    : 0.14 s -- 21 partitions

Règle :
  Augmenter ou équilibrer les partitions -> repartition() (shuffle)
  Réduire les partitions avant une écriture -> coalesce() (pas de shuffle)


---
## 2.4 Lecture avancée du Spark UI

Le Spark UI est votre principal outil de diagnostic. Voici les indicateurs
à surveiller dans chaque onglet.

### Onglet Jobs
- **Duration** : durée totale. Si un job est anormalement long, aller dans Stages.
- **Stages Skipped** : stages dont le résultat était en cache -- c'est bon signe.

### Onglet Stages
- **Task Distribution** : si la barre des durées est très étalée, il y a du skew.
- **Input / Shuffle Read / Shuffle Write** : un Shuffle Write élevé indique
  un shufflecoûteux. Chercher à le réduire par repartitionnement ou broadcast.
- **Spill (Memory / Disk)** : si non nul, Spark a dû écrire sur disque faute
  de mémoire. Augmenter `spark.driver.memory` ou réduire la taille des partitions.

### Onglet Storage
- Les DataFrames en cache apparaissent ici avec leur taille en mémoire et sur disque.
- Si **Fraction Cached < 1.0**, le cache a débordé. Utiliser `MEMORY_AND_DISK`.

### Onglet SQL
- Chaque requête DataFrame ou SQL génère un plan visualisé en DAG.
- Les noeuds en **orange** sont les shuffles (exchange operators).
- Les **métriques de chaque noeud** (lignes lues, lignes émises) permettent
  d'identifier les filtres peu sélectifs ou les jointures cartésiennes accidentelles.


In [37]:
# Génération d'un job intentionnellement lent pour analyse dans le Spark UI
print("Génération d'un job avec shuffle visible dans le Spark UI...")
print("Ouvrez http://localhost:4040 -> SQL/DataFrame -> dernière requête")

df_analyse = (
    df
    .groupBy("station_id", "annee", "mois", "heure")
    .agg(
        spark_avg("taux_occupation").alias("taux_moy"),
        F.stddev("taux_occupation").alias("taux_std"),
        F.count("*").alias("n")
    )
    .filter(col("n") >= 10)
    .join(
        df.groupBy("station_id")
          .agg(spark_avg("taux_occupation").alias("taux_global")),
        on="station_id"
    )
    .withColumn("ecart_global", col("taux_moy") - col("taux_global"))
    .orderBy(F.desc("ecart_global"))
)

t0 = time.perf_counter()
n = df_analyse.count()
print(f"Résultat : {n:,} lignes en {time.perf_counter()-t0:.2f} s")
print("\nAllez maintenant dans Spark UI -> SQL/DataFrame -> dernière entrée.")
print("Identifiez : le nombre d'Exchange (shuffle), l'opération la plus coûteuse.")


Génération d'un job avec shuffle visible dans le Spark UI...
Ouvrez http://localhost:4040 -> SQL/DataFrame -> dernière requête


Résultat : 196,072 lignes en 0.76 s

Allez maintenant dans Spark UI -> SQL/DataFrame -> dernière entrée.
Identifiez : le nombre d'Exchange (shuffle), l'opération la plus coûteuse.


### Lecture du Spark UI par programme (analyse chiffrée des stages)

Le Spark UI expose aussi ses données en JSON (API REST, `http://localhost:4040/api/v1/...`).
On s'en sert ici pour **lister les stages les plus longs** de la session et repérer un éventuel
*skew* (grand écart entre la tâche médiane et la tâche la plus lente) ou un *shuffle* coûteux,
sans dépendre d'une capture d'écran.

In [38]:
def stages_les_plus_longs(n: int = 5) -> list[dict]:
    """Interroge l'API REST du Spark UI et renvoie les `n` stages les plus longs.

    Args:
        n: Nombre de stages à retenir.

    Returns:
        Liste de dictionnaires (identifiant, nom, tâches, durée cumulée, volumes lus / écrits en
        shuffle, ratio max / médiane des durées de tâches), du plus long au plus court.
        Liste vide si le Spark UI est injoignable.
    """
    base = f"http://localhost:4040/api/v1/applications/{sc.applicationId}"
    try:
        stages = requests.get(f"{base}/stages", params={"status": "complete"}, timeout=10).json()
    except requests.RequestException as e:
        print(f"[INFO] Spark UI injoignable : {e}")
        return []
    plus_longs = sorted(stages, key=lambda s: s.get("executorRunTime", 0), reverse=True)[:n]
    resultats = []
    for s in plus_longs:
        ratio = None
        try:
            r = requests.get(f"{base}/stages/{s['stageId']}/{s['attemptId']}/taskSummary",
                             params={"quantiles": "0.5,1.0"}, timeout=10).json()
            mediane, maxi = r["executorRunTime"]
            ratio = round(maxi / mediane, 1) if mediane else None
        except (requests.RequestException, KeyError, ValueError):
            pass
        resultats.append({
            "stage": s["stageId"], "nom": s["name"][:45], "taches": s["numCompleteTasks"],
            "duree_cumulee_s": round(s["executorRunTime"] / 1000, 1),
            "shuffle_lu_Mo": round(s.get("shuffleReadBytes", 0) / 1_048_576, 1),
            "shuffle_ecrit_Mo": round(s.get("shuffleWriteBytes", 0) / 1_048_576, 1),
            "ratio_max_mediane": ratio,
        })
    return resultats

top_stages = stages_les_plus_longs(6)
print(pd.DataFrame(top_stages).to_string(index=False))
MESURES["spark_ui_stages"] = top_stages
# Lecture : un ratio max/médiane proche de 1 = tâches équilibrées ; nettement supérieur à 1 = une
# tâche traîne (skew). Un « shuffle_ecrit » élevé signale un shuffle coûteux à réduire
# (broadcast, repartitionnement, agrégation partielle).

 stage                                           nom  taches  duree_cumulee_s  shuffle_lu_Mo  shuffle_ecrit_Mo  ratio_max_mediane
 18778 javaToPython at NativeMethodAccessorImpl.java      21            267.1            0.0             921.9              241.0
 18783                          count at <unknown>:0      21              7.4            0.0               1.9                1.6
 18707                          count at <unknown>:0      24              6.3            0.0               0.0                1.2
 18687 showString at NativeMethodAccessorImpl.java:0      48              4.2            0.0               0.0                1.8
 18715                          count at <unknown>:0      24              3.7            0.0               0.0                1.6
 18592          treeAggregate at Statistics.scala:58       8              3.4            0.0               0.0                1.0


---
## 2.5 Quand utiliser Spark -- et quand ne pas l'utiliser ?

Spark n'est pas la bonne réponse à tous les problèmes. Voici un guide de décision
construit à partir des expériences de ces trois jours.

| Critère | Pandas | PySpark |
|---------|--------|---------|
| Volume | < 10 GB en RAM | > 10 GB ou hors RAM |
| Latence requise | < 1 s (interactif) | secondes à minutes (batch) |
| Infrastructure | laptop / serveur seul | cluster ou machine puissante |
| Complexité d'installation | `pip install pandas` | JVM + config |
| Débogage | simple (Python pur) | plus complexe (Spark UI) |
| Streaming | non (ou limité) | oui (Structured Streaming) |
| ML distribué | scikit-learn | MLlib |
| SQL analytique | limité | natif et optimisé |

**Règle pratique** : commencez par Pandas. Si vous rencontrez des problèmes
de mémoire, de temps de calcul, ou si vous avez besoin de streaming ou de
distribution, migrez vers Spark -- généralement en changeant `pd.read_*`
en `spark.read.*` et en adaptant les agrégations.


In [39]:
# Mesure finale comparative : même calcul en Pandas vs Spark
# (sur un sous-ensemble pour que Pandas puisse le tenir en mémoire)
# CONTRAINTE n°4 : toPandas() n'accepte que moins de 10 000 lignes. Le modèle prélevait 10 %
# de 2023 (des centaines de milliers de lignes) ; je prélève ici ~9 000 lignes.
df_2021 = df.filter(col("annee") == 2021)
n_2021 = df_2021.count()
df_sample_sp = df_2021.sample(fraction=9_000 / n_2021, seed=SEED).cache()
df_sample_pd = df_sample_sp.toPandas()
assert len(df_sample_pd) < 10_000, "Trop de lignes pour toPandas() (contrainte n°4)"
print(f"Sous-échantillon Pandas : {len(df_sample_pd):,} lignes")

# ── Pandas ────────────────────────────────────────────────────────────────────
import pandas as pd

t0 = time.perf_counter()
result_pd = (
    df_sample_pd
    .groupby(["station_id", "heure"])["taux_occupation"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "taux_moy", "std": "taux_std", "count": "n"})
    .query("n >= 2")
    .sort_values("taux_moy", ascending=False)
)
t_pandas = time.perf_counter() - t0

# ── Spark ─────────────────────────────────────────────────────────────────────
def agregation_spark(donnees, seuil):
    """Moyenne, écart-type et effectif du taux d'occupation par (station, heure).

    Args:
        donnees: DataFrame Spark avec ``station_id``, ``heure``, ``taux_occupation``.
        seuil: Effectif minimal d'un groupe pour être conservé.

    Returns:
        DataFrame Spark agrégé, trié par taux moyen décroissant.
    """
    return (
        donnees
        .groupBy("station_id", "heure")
        .agg(
            spark_avg("taux_occupation").alias("taux_moy"),
            F.stddev("taux_occupation").alias("taux_std"),
            F.count("*").alias("n"),
        )
        .filter(col("n") >= seuil)
        .orderBy(F.desc("taux_moy"))
    )

# (a) même petit échantillon que Pandas : comparaison à armes égales
t0 = time.perf_counter()
n_sp_petit = agregation_spark(df_sample_sp, 2).count()
t_spark_petit = time.perf_counter() - t0

# (b) toutes les données 2021 : là où Pandas n'a pas le droit d'aller ici
t0 = time.perf_counter()
n_sp_grand = agregation_spark(df_2021, 2).count()
t_spark_grand = time.perf_counter() - t0

print(f"\nPandas, {len(df_sample_pd):,} lignes : {t_pandas:.3f} s  ({len(result_pd):,} groupes)")
print(f"Spark,  {len(df_sample_pd):,} lignes : {t_spark_petit:.3f} s  ({n_sp_petit:,} groupes)")
print(f"Spark,  {n_2021:,} lignes : {t_spark_grand:.3f} s  ({n_sp_grand:,} groupes)")
print()
print("Conclusion : sur ~9 000 lignes, Pandas est plus rapide (pas d'ordonnancement, de sérialisation")
print("JVM ni de shuffle). Spark paie un surcoût fixe, mais il traite")
print(f"{n_2021 / len(df_sample_pd):,.0f} fois plus de lignes en {t_spark_grand / t_spark_petit:.0f} fois plus de temps seulement.")
print("Spark devient intéressant quand le volume dépasse la RAM disponible,")
print("ou quand on intègre batch + streaming + ML dans le même pipeline.")
MESURES["pandas_vs_spark"] = {"lignes_petit": len(df_sample_pd), "pandas_petit_s": t_pandas,
    "spark_petit_s": t_spark_petit, "lignes_grand": n_2021, "spark_grand_s": t_spark_grand}

Sous-échantillon Pandas : 9,200 lignes



Pandas, 9,200 lignes : 0.010 s  (1,099 groupes)
Spark,  9,200 lignes : 0.080 s  (1,099 groupes)
Spark,  7,470,645 lignes : 0.348 s  (33,144 groupes)

Conclusion : sur ~9 000 lignes, Pandas est plus rapide (pas d'ordonnancement, de sérialisation
JVM ni de shuffle). Spark paie un surcoût fixe, mais il traite
812 fois plus de lignes en 4 fois plus de temps seulement.
Spark devient intéressant quand le volume dépasse la RAM disponible,
ou quand on intègre batch + streaming + ML dans le même pipeline.


---
## 2.6 Pipeline final bout-en-bout

Pour clore le cours, nous assemblons en un seul pipeline la chaîne complète :
ingestion Delta → feature engineering SQL → prédiction MLlib → résumé analytique.

C'est la démonstration qu'un pipeline Spark couvre l'intégralité du cycle
de vie de la donnée, du stockage brut à la valeur métier.


In [40]:
from pyspark.ml import PipelineModel

print("=== Pipeline final ClimaCity Paris ===\n")
# ── 1. Ingestion depuis Delta ─────────────────────────────────────────────────
# « Nouveau batch » : mars-avril 2021, la période du jeu de test (le modèle n'a jamais vu ces
# données pendant son entraînement). Les lignes fictives de 2022 sont exclues.
print("[1/5] Lecture depuis Delta Lake...")
t0 = time.perf_counter()
df_ingere = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE))
    .filter((col("annee") == 2021) & (col("mois") >= 3))
)
print(f"      {df_ingere.count():,} snapshots chargés en {time.perf_counter()-t0:.1f}s")

# ── 2. Feature engineering ────────────────────────────────────────────────────
# ADAPTATION : le modèle dupliquait en SQL la construction des variables et fixait cluster=0
# pour toutes les stations, alors que l'entraînement utilisait les vrais clusters. Je réutilise
# ici la fonction `ajouter_features` du carnet 5 : entraînement et production calculent les
# mêmes variables (principe DRY), et je rejoins les vrais clusters.
print("[2/5] Feature engineering (fonction partagée ajouter_features + clusters)...")
df_features_final = (
    ajouter_features(df_ingere)
    .join(df_clusters, on="station_id", how="left")
    .fillna(0, subset=["cluster"])          # station sans profil : cluster par défaut, comme à l'entraînement
    .dropna(subset=FEATURES)
)
print(f"      {df_features_final.count():,} lignes avec features complètes")

# ── 3. Prédiction avec le modèle MLlib ────────────────────────────────────────
print("[3/5] Application du modèle GBT...")
model_prod = PipelineModel.load(str(chemin_model_local))     # rechargé depuis le disque
df_predictions = model_prod.transform(df_features_final)
print(f"      {df_predictions.count():,} prédictions générées")

# ── 4. Identification des stations à risque (taux prédit < 0.15) ──────────────
print("[4/5] Identification des stations à risque...")
df_risque = (
    df_predictions
    .filter(col("prediction") < 0.15)
    .groupBy("station_id", "nom_station", "code_arr", "heure")
    .agg(
        spark_round(spark_avg("prediction"), 3).alias("taux_predit_moy"),
        F.count("*").alias("nb_occurrences")
    )
    .orderBy("taux_predit_moy")
)
print(f"      {df_risque.count()} combinaisons station×heure à risque (taux prédit < 15%)")

# ── 5. Écriture du rapport en Delta ───────────────────────────────────────────
print("[5/5] Écriture du rapport...")
chemin_rapport = OUTPUT_DIR / "delta" / "rapport_risque"
(
    df_risque.write
    .format("delta")
    .mode("overwrite")
    .save(str(chemin_rapport))
)
print(f"      Rapport écrit dans {chemin_rapport}")

print("\n=== Pipeline terminé avec succès ===")

=== Pipeline final ClimaCity Paris ===

[1/5] Lecture depuis Delta Lake...
      3,801,377 snapshots chargés en 0.1s
[2/5] Feature engineering (fonction partagée ajouter_features + clusters)...


      2,518,700 lignes avec features complètes
[3/5] Application du modèle GBT...


      2,518,700 prédictions générées
[4/5] Identification des stations à risque...


      23096 combinaisons station×heure à risque (taux prédit < 15%)
[5/5] Écriture du rapport...


      Rapport écrit dans ../data/output/delta/rapport_risque

=== Pipeline terminé avec succès ===


In [41]:
# Lecture et affichage du rapport final
df_rapport = spark.read.format("delta").load(str(OUTPUT_DIR / "delta" / "rapport_risque"))

print("Top 20 des situations à risque (station × heure avec taux prédit le plus bas) :")
df_rapport.orderBy("taux_predit_moy").show(20, truncate=False)

print(f"\nRésumé par arrondissement :")
(
    df_rapport
    .groupBy("code_arr")
    .agg(
        F.count("*").alias("nb_situations_risque"),
        spark_round(spark_avg("taux_predit_moy"), 3).alias("taux_moyen_risque")
    )
    .orderBy(F.desc("nb_situations_risque"))
    .show(20)
)


Top 20 des situations à risque (station × heure avec taux prédit le plus bas) :


+----------+--------------------------------------+--------+-----+---------------+--------------+
|station_id|nom_station                           |code_arr|heure|taux_predit_moy|nb_occurrences|
+----------+--------------------------------------+--------+-----+---------------+--------------+
|2339      |Université                            |92      |5    |0.02           |6             |
|2275      |Square Pierre Lazareff                |2       |0    |0.022          |4             |
|1408      |François Truffaut - Saint Emilion     |12      |4    |0.024          |1             |
|1454      |Gare d'Austerlitz - Quai Saint-Bernard|13      |3    |0.025          |2             |
|2383      |Vincent Auriol - Louise Weiss         |13      |3    |0.025          |6             |
|2181      |Roquépine - Malesherbes               |8       |3    |0.025          |6             |
|2174      |Rome - Provence                       |8       |3    |0.025          |4             |
|2127      |Quai de 

---
## 2.7 Bilan d'architecture : quand Spark vaut-il son overhead ?

Toutes les mesures prises pendant le projet (carnets 2, 5 et 6) sont réunies dans
`data/output/mesures.json`. La cellule suivante les affiche ; les conclusions s'appuient sur elles.

In [42]:
mesures.enregistrer(MESURES_JSON, "optimisation", MESURES)      # mesures de ce carnet
tout = mesures.lire(MESURES_JSON)

def ligne(nom: str, valeur) -> None:
    """Affiche une mesure alignée."""
    print(f"  {nom:<48} {valeur}")

print("=== Mesures du projet ===")
c = tout.get("cache_jointure")
if c:
    print("Cache sur la jointure répétée (carnet 2) :")
    ligne("sans cache (moyenne / passe)", f"{sum(c['sans_cache_s'])/len(c['sans_cache_s']):.2f} s")
    ligne("avec cache (moyenne / passe)", f"{sum(c['avec_cache_s'])/len(c['avec_cache_s']):.2f} s")
    ligne("gain", f"x{c['gain']:.1f}")
o = tout["optimisation"]
b = o["broadcast"]
print("Jointure broadcast (carnet 6) :")
ligne("SortMergeJoin", f"{b['smj_s']:.2f} s, {b['shuffles_smj']} shuffle(s)")
ligne("BroadcastHashJoin", f"{b['broadcast_s']:.2f} s, {b['shuffles_broadcast']} shuffle(s)")
ligne("gain", f"x{b['gain']:.1f}")
print("Nombre de partitions de shuffle (calcul itératif simulé) :")
ligne("durées (s) par nombre de partitions", o["partitions_shuffle_s"])
p = o["pandas_vs_spark"]
print("Pandas vs Spark :")
ligne(f"Pandas ({p['lignes_petit']:,} lignes)", f"{p['pandas_petit_s']:.3f} s")
ligne(f"Spark  ({p['lignes_petit']:,} lignes)", f"{p['spark_petit_s']:.3f} s")
ligne(f"Spark  ({p['lignes_grand']:,} lignes)", f"{p['spark_grand_s']:.3f} s")
m = tout.get("ml")
if m:
    print("Apprentissage (carnet 5) :")
    ligne("GBT (fit)", f"{m['fit_gbt_s']} s sur {m['n_train_gbt']:,} lignes")
    ligne("CrossValidator", f"{m['cv_s']} s")
    ligne("RMSE / R² du modèle initial", f"{m['rmse_initial']:.4f} / {m['r2_initial']:.3f}")

=== Mesures du projet ===
Cache sur la jointure répétée (carnet 2) :
  sans cache (moyenne / passe)                     3.30 s
  avec cache (moyenne / passe)                     0.16 s
  gain                                             x20.1
Jointure broadcast (carnet 6) :
  SortMergeJoin                                    1.09 s, 2 shuffle(s)
  BroadcastHashJoin                                0.37 s, 0 shuffle(s)
  gain                                             x3.0
Nombre de partitions de shuffle (calcul itératif simulé) :
  durées (s) par nombre de partitions              {'2': 0.45, '4': 0.2, '8': 0.13, '16': 0.17, '32': 0.15}
Pandas vs Spark :
  Pandas (9,200 lignes)                            0.010 s
  Spark  (9,200 lignes)                            0.080 s
  Spark  (7,470,645 lignes)                        0.348 s
Apprentissage (carnet 5) :
  GBT (fit)                                        26.3 s sur 1,995,716 lignes
  CrossValidator                                   101.0 s

### Conclusions d'architecture (appuyées sur les mesures ci-dessus)

1. **Spark a un coût fixe.** Sur ~9 000 lignes, Pandas répond en quelques millisecondes ; Spark est
   nettement plus lent (planification du job, sérialisation, ordonnancement des tâches). À cette échelle, **Pandas gagne**.
2. **Mais Spark ne s'effondre pas quand le volume grandit.** Le même calcul sur les ~7,5 millions de lignes
   de 2021 prend un temps multiplié par quelques unités seulement pour un volume multiplié par ~800 (voir les
   mesures), là où Pandas, avec le même code, demanderait plusieurs Go de RAM. Le seuil de bascule est donc celui où les données ne tiennent plus confortablement en mémoire
   d'une machine, ou celui où l'on veut **enchaîner batch, streaming et ML** dans un même outil
   (c'est le cas de tout ce projet).
3. **Les optimisations comptent plus que la taille du cluster.** Le cache (jointure répétée), le
   broadcast (un shuffle en moins) et le bon nombre de partitions ont un effet mesurable sur une seule
   machine. Ils se lisent dans le Spark UI, d'où l'intérêt de l'analyse par stages ci-dessus.
4. **Un diagnostic avant une technique.** Le salting du carnet n'a apporté qu'un gain marginal, dans le bruit
   de mesure, sur une seule machine : sans skew avéré (ratio max/médiane élevé dans le Spark UI), on ne
   l'applique pas. À noter : le stage le plus long de la session (`javaToPython`) affiche un ratio
   max/médiane très supérieur à 1 ; son nom (transfert JVM vers Python) ne correspond pas à une agrégation
   par clé, mais nous ne l'avons pas investigué plus avant : à examiner dans le Spark UI avant toute conclusion.

---
## Bilan du Jour 3 et du projet

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Features temporelles cycliques | MLlib / DataFrame | Encodage sin/cos, features de lag |
| Split temporel train/test | MLlib | Pas de fuite d'information |
| Clustering K-Means | MLlib | Pipeline, méthode du coude, `VectorAssembler` |
| Visualisation Folium | Python | Carte interactive des clusters |
| Régression GBT | MLlib | `GBTRegressor`, importance des features |
| Évaluation | MLlib | RMSE, MAE, R², `RegressionEvaluator` |
| Validation croisée | MLlib | `CrossValidator`, `ParamGridBuilder` |
| Tracking d'expériences | MLflow | `log_params`, `log_metrics`, `log_model` |
| Rechargement de modèle | MLflow / Spark | `mlflow.spark.load_model`, `PipelineModel.load` |
| Analyse du Catalyst | Spark | `explain(mode="formatted")`, plans logique/physique |
| Data skew | Spark | Salting, diagnostic Spark UI |
| Broadcast join | Spark | `broadcast()`, seuil automatique |
| Partitionnement | Spark | `repartition` vs `coalesce`, `shuffle.partitions` |
| Pipeline bout-en-bout | Spark | Delta → SQL → MLlib → Delta |

### Ce que vous savez faire après ces trois jours

À l'issue de ce projet, vous maîtrisez l'ensemble du spectre Spark :

- **Ingestion** : RDD, DataFrame, lecture Parquet/Delta avec predicate pushdown.
- **Transformation** : API DataFrame, Spark SQL, fenêtrage analytique, jointures.
- **Persistance** : Delta Lake ACID, time-travel, MERGE INTO.
- **Streaming** : source fichier, fenêtres glissantes, watermark, foreachBatch.
- **Machine Learning** : Pipeline MLlib, clustering, régression, validation croisée.
- **Observabilité** : Spark UI, MLflow, `explain()`.
- **Optimisation** : cache, broadcast, repartitionnement, diagnostic du skew.

### Prochaines étapes

Ce projet pose les bases. Pour aller plus loin :

- **Kafka** : remplacer la file source par un vrai broker pour le streaming.
- **Kubernetes / Databricks** : déployer sur un vrai cluster.
- **MLflow Model Registry** : versionner et promouvoir les modèles en production.
- **Delta Live Tables** : orchestration déclarative de pipelines Delta.
- **GraphFrames** : analyser le réseau de stations comme un graphe.


In [43]:
spark.stop()
print("SparkSession arrêtée. Projet ClimaCity Paris terminé !")


SparkSession arrêtée. Projet ClimaCity Paris terminé !
